# 🎬 ASTRAA — LTX-Video Colab
Free/open-source image-to-video test for ASTRAA.

## 1. Check GPU
Run this first. A free T4 is ideal for this lightweight test.

In [ ]:
!nvidia-smi


## 2. Install LTX-Video
Uses the official Lightricks repository.

In [ ]:
%cd /content
!rm -rf LTX-Video
!git clone --depth 1 https://github.com/Lightricks/LTX-Video.git
!cd /content/LTX-Video && git fetch --depth 1 origin bdc8f017f0148a0f0bb9e3a5049d2d356423cee0 && git checkout bdc8f017f0148a0f0bb9e3a5049d2d356423cee0
%cd /content/LTX-Video
!pip install -q -e '.[inference]'
!pip install -q --force-reinstall --no-deps 'huggingface-hub~=0.30'


## 3. Download the lighter 2B distilled model
This is intended for lighter VRAM than the 13B model.

In [ ]:
from huggingface_hub import hf_hub_download
model_dir='/content/LTX-Video/models'
import os
os.makedirs(model_dir, exist_ok=True)
hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltxv-2b-0.9.8-distilled.safetensors', local_dir=model_dir)
print('Model downloaded:', model_dir)


## 4. Upload the ASTRAA reference image
Upload one image such as Aarav + Maa Meera. Keep the image in `/content/LTX-Video/`.

In [ ]:
from google.colab import files
uploaded=files.upload()
image_name=next(iter(uploaded))
image_path=f'/content/LTX-Video/{image_name}'
print(image_path)


## 5. Generate the first test shot
Start with an ultra-small 9-frame T4 test. This first run uses CPU offload to avoid a T4 runtime crash.


In [ ]:
import os, subprocess, yaml, glob

# Extra-safe memory settings for a 15 GB T4.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

base = '/content/LTX-Video'
os.makedirs(f'{base}/models', exist_ok=True)
os.makedirs(f'{base}/configs', exist_ok=True)

# Get the verified LTX 0.9.8 inference script and config.
for name in ['inference.py', 'configs/ltxv-2b-0.9.8-distilled.yaml']:
    path = f'{base}/{name}'
    if not os.path.isfile(path):
        url = f'https://raw.githubusercontent.com/Lightricks/LTX-Video/main/{name}'
        r = subprocess.run(['wget', '-q', '-O', path, url], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f'Could not download {name}: {r.stderr}')

model = f'{base}/models/ltxv-2b-0.9.8-distilled.safetensors'
if not os.path.isfile(model):
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id='Lightricks/LTX-Video',
        filename='ltxv-2b-0.9.8-distilled.safetensors',
        local_dir=f'{base}/models'
    )

# IMPORTANT: the official script moves all models to the GPU during startup.
# On a 15 GB T4 that can crash the Colab runtime before normal offload starts.
# Patch only that startup behavior: keep weights on CPU and use Diffusers offload.
inference_path = f'{base}/inference.py'
src = open(inference_path, encoding='utf-8').read()

old_startup = """    transformer = transformer.to(device)
    vae = vae.to(device)
    text_encoder = text_encoder.to(device)
"""
new_startup = """    # Keep components on CPU during construction; moving all of them to a T4 at once can OOM.
"""
if old_startup not in src:
    raise RuntimeError('Expected LTX startup block was not found; refusing to patch the wrong code.')
src = src.replace(old_startup, new_startup, 1)

old_pipeline = """    pipeline = LTXVideoPipeline(**submodel_dict)
    pipeline = pipeline.to(device)
    return pipeline
"""
new_pipeline = """    pipeline = LTXVideoPipeline(**submodel_dict)
    if device.type == "cuda":
        pipeline.enable_model_cpu_offload(device=device)
    else:
        pipeline = pipeline.to(device)
    return pipeline
"""
if old_pipeline not in src:
    raise RuntimeError('Expected LTX pipeline block was not found; refusing to patch the wrong code.')
src = src.replace(old_pipeline, new_pipeline, 1)

open(inference_path, 'w', encoding='utf-8').write(src)

# Build a single-scale config and disable prompt enhancement/upscaling.
with open(f'{base}/configs/ltxv-2b-0.9.8-distilled.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['checkpoint_path'] = model
cfg.pop('pipeline_type', None)
cfg.pop('spatial_upscaler_model_path', None)
cfg['prompt_enhancement_words_threshold'] = 0
local_cfg = f'{base}/configs/astraa-2b-t4-safe.yaml'
with open(local_cfg, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# Find the uploaded reference image.
image_candidates = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
    image_candidates.extend(glob.glob(f'{base}/{ext}'))

if 'image_path' not in globals() or not os.path.isfile(image_path):
    if not image_candidates:
        raise FileNotFoundError('No reference image found. Run Step 4 once to upload one.')
    image_path = image_candidates[0]

print('Using reference image:', image_path)
print('READY:', os.path.isfile(inference_path), os.path.isfile(local_cfg), os.path.isfile(model))

PROMPT = (
    "Aarav's mother Meera remains visually consistent with the reference image. "
    "She is inside her house at night. The curtains gently move in the wind. "
    "She slowly looks toward the entrance with a worried protective expression. "
    "Slow cinematic camera push-in, subtle dust particles, dramatic nighttime lighting, "
    "high-quality 3D animated Indian fantasy movie style, natural motion, no dialogue, no text."
)

result = subprocess.run([
    'python', inference_path,
    '--prompt', PROMPT,
    '--conditioning_media_paths', image_path,
    '--conditioning_start_frames', '0',
    '--height', '128',
    '--width', '192',
    '--num_frames', '9',
    '--seed', '42',
    '--pipeline_config', local_cfg,
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'LTX inference failed with exit code {result.returncode}')
print('SUCCESS: first T4 test shot finished.')


In [ ]:
import os, glob
videos=glob.glob('/content/LTX-Video/**/*.mp4', recursive=True)
print('\n'.join(videos[-10:]) if videos else 'No MP4 found yet.')


## Next
Once the first shot works, we will add reusable ASTRAA prompts and an extension workflow.